# Flywheel Results
Track improvement across flywheel cycles. Did the loop actually work? Which models got promoted, and by how much did they improve over the baseline?

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv('../.env')
mongo = MongoClient(os.getenv('MONGO_URI', 'mongodb://localhost:27017'))
db = mongo[os.getenv('MONGO_DB', 'data_flywheel')]
print('Connected')

## 1. Flywheel Run History

In [ ]:
runs = list(db.flywheel_runs.find().sort('started_at', 1))

if not runs:
    print('No runs yet — trigger one with: make run-flywheel')
else:
    rows = []
    for r in runs:
        stages = r.get('stages', {})
        curation = stages.get('curation', {})
        promotion = stages.get('promotion', {})
        rows.append({
            'run_id': r['_id'][:8],
            'started_at': r['started_at'],
            'status': r['status'],
            'triggered_by': r.get('triggered_by', 'manual'),
            'logs_pulled': curation.get('logs_pulled', 0),
            'samples_curated': curation.get('samples_after_dedup', 0),
            'promoted': promotion.get('promoted', False),
            'promoted_model': promotion.get('promoted_model_name'),
        })

    df_runs = pd.DataFrame(rows)
    print(f'Total runs: {len(df_runs)}')
    print(f'Promotions: {df_runs["promoted"].sum()}')
    display(df_runs)

## 2. Dataset Size Per Cycle

In [ ]:
if runs:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    run_labels = df_runs['run_id']
    x = range(len(df_runs))

    axes[0].bar(x, df_runs['logs_pulled'], label='Logs pulled', color='steelblue', alpha=0.8)
    axes[0].bar(x, df_runs['samples_curated'], label='After curation', color='coral', alpha=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(run_labels, rotation=30, ha='right')
    axes[0].set_title('Logs Pulled vs Curated Samples per Cycle')
    axes[0].set_ylabel('Count')
    axes[0].legend()

    # Curation retention rate
    df_runs['retention'] = df_runs['samples_curated'] / df_runs['logs_pulled'].replace(0, np.nan)
    axes[1].plot(x, df_runs['retention'], marker='o', color='steelblue')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(run_labels, rotation=30, ha='right')
    axes[1].set_title('Curation Retention Rate per Cycle')
    axes[1].set_ylabel('Fraction kept')
    axes[1].set_ylim(0, 1)
    axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

    plt.tight_layout()
    plt.show()

## 3. Model Registry — Promotion History

In [ ]:
registry = list(db.model_registry.find().sort('created_at', 1))

if not registry:
    print('No models in registry yet')
else:
    df_reg = pd.DataFrame([{
        'model_id': r['_id'][:8],
        'model_name': r['model_name'],
        'status': r['status'],
        'accuracy': r.get('metrics', {}).get('accuracy'),
        'latency_p95_ms': r.get('metrics', {}).get('latency_p95_ms'),
        'cost_per_1k': r.get('metrics', {}).get('cost_per_1k_tokens'),
        'promoted_at': r.get('promoted_at'),
    } for r in registry])

    print(f'Total models: {len(df_reg)}')
    print(f'Current production: {df_reg[df_reg["status"]=="production"]["model_name"].values}')
    display(df_reg)

## 4. Accuracy Improvement Across Cycles

In [ ]:
if registry and len(df_reg.dropna(subset=['accuracy'])) > 1:
    promoted = df_reg[df_reg['promoted_at'].notna()].sort_values('promoted_at')

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    metrics = [
        ('accuracy', 'Accuracy (win rate)', True),
        ('latency_p95_ms', 'Latency p95 (ms)', False),
        ('cost_per_1k', 'Cost per 1k tokens (USD)', False),
    ]

    for ax, (col, label, higher_is_better) in zip(axes, metrics):
        if col in promoted.columns and promoted[col].notna().any():
            ax.plot(range(len(promoted)), promoted[col], marker='o',
                    color='steelblue' if higher_is_better else 'coral',
                    linewidth=2, markersize=8)
            ax.set_xticks(range(len(promoted)))
            ax.set_xticklabels(
                [f"{row['model_name']}" for _, row in promoted.iterrows()],
                rotation=20, ha='right', fontsize=8
            )
            ax.set_title(label)
            direction = '↑ better' if higher_is_better else '↓ better'
            ax.set_ylabel(f'{label} ({direction})')

    plt.suptitle('Promoted Model Metrics Across Cycles', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Need at least 2 promotion events to show trend — run more cycles')

## 5. Run Status Summary

In [ ]:
if runs:
    status_counts = df_runs['status'].value_counts()
    trigger_counts = df_runs['triggered_by'].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    colors = {'completed': 'steelblue', 'failed': 'salmon', 'pending': 'gray',
              'curating': 'orange', 'evaluating': 'purple'}
    bar_colors = [colors.get(s, 'steelblue') for s in status_counts.index]

    axes[0].bar(status_counts.index, status_counts.values, color=bar_colors, edgecolor='white')
    axes[0].set_title('Run Status Distribution')
    axes[0].set_ylabel('Count')

    axes[1].pie(trigger_counts.values, labels=trigger_counts.index,
                autopct='%1.1f%%', colors=['steelblue', 'coral'],
                startangle=90)
    axes[1].set_title('Trigger Source\n(manual vs celery_beat)')

    plt.tight_layout()
    plt.show()